In [ ]:
# CELL 1: Install packages (run once, will crash kernel — that's normal!)
!pip install -q sympy numpy transformers accelerate tqdm antlr4-python3-runtime==4.11.1 vllm
import os
os.kill(os.getpid(), 9)

In [ ]:
# CELL 2: Extra installs (run after kernel crash)
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
    !uv pip install -qqq --no-deps --upgrade "torchao>=0.16.0"
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

In [ ]:
# CELL 3: Imports and config
import json
import os
import re
import sys
import shutil
import torch
from pathlib import Path
from typing import Optional
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
DATA_PATH = "data/public.jsonl"
OUTPUT_PATH = "results/prompt_improved_results.jsonl"
MAX_TOKENS  = 4192


os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")

In [ ]:
import os, shutil
os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

# Data: put the question file at data/public.jsonl (judger.py lives in the repo root)
assert os.path.exists("data/public.jsonl"), "Put the dataset at data/public.jsonl first"
assert os.path.exists("judger.py"), "Run this notebook from the repo root so judger.py is importable"

data = [__import__('json').loads(line) for line in open(DATA_PATH)]
print(f"Loaded {len(data)} questions")

In [ ]:
# CELL 5: Prompts and build_prompt function
SYSTEM_PROMPT_MCQ = (
    "You are a world-class mathematician. "
    "This is a multiple choice question - select the best answer. "
    "Reason briefly. "
    "You MUST ALWAYS end your response with \\boxed{X} where X is your chosen letter. "
    "Always attempt the problem even if you are unsure. Never say you cannot solve it. "
    "After reaching a conclusion, immediately write \\boxed{} without additional commentary."
)

SYSTEM_PROMPT_MATH = (
    "You are a world-class mathematician. "
    "Solve the problem step by step. "
    "Count the number of [ANS] placeholders in the question - you must provide exactly that many answers. "
    "Separate multiple answers by commas inside a single \\boxed{}, e.g. \\boxed{3, 7}. "
    "You MUST ALWAYS end your response with \\boxed{}. "
    "Always attempt the problem even if you are unsure. Never say you cannot solve it. "
    "After reaching a conclusion, immediately write \\boxed{} without additional commentary."
)

OTHER = (
    "You are a world-class mathematician. "
    "This question has answer choices embedded in the text. "
    "Reason briefly. "
    "Select the correct letter(s) and put them inside \\boxed{}. "
    "If multiple answers are correct write them together like \\boxed{BC}. "
    "You MUST ALWAYS end your response with \\boxed{}. "
    "Always attempt the problem even if you are unsure. Never say you cannot solve it. "
    "After reaching a conclusion, immediately write \\boxed{} without additional commentary."
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    elif options is None and ('A.' in question or 'B.' in question):
        return OTHER, question
    return SYSTEM_PROMPT_MATH, question

In [ ]:
# CELL 6: Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side='left')
tokenizer.pad_token = tokenizer.eos_token
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
)

In [ ]:
# CELL 7: Resume logic + build prompts
batch_size = 2
prompts = []
items_to_process = []

from google.colab import drive
drive.mount('/content/drive')
LOCAL_BACKUP = "/content/drive/MyDrive/prompt_improved_backup.jsonl"
if not os.path.exists(OUTPUT_PATH) and os.path.exists(LOCAL_BACKUP):
    shutil.copy(LOCAL_BACKUP, OUTPUT_PATH)
    print("Restored from backup!")

# 2. RESUME LOGIC
completed_ids = set()
if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH) as f:
        for line in f:
            r = json.loads(line)
            completed_ids.add(r["id"])
print(f"Resuming: {len(completed_ids)} already done")

# 3. BUILD PROMPTS, SKIPPING COMPLETED
for item in data:
    if item['id'] in completed_ids:
        continue
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)
    items_to_process.append(item)

print(f"Generating responses for {len(prompts)} questions...")
print(f"Batch size: {batch_size} | Total batches: {len(range(0, len(prompts), batch_size))}")

In [ ]:
# CELL 8: Generation + scoring + saving
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""

def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()

results = []

for i in tqdm(range(0, len(prompts), batch_size), desc="Generating"):
    # 1. TOKENIZE
    inputs = tokenizer(
        prompts[i:i+batch_size],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048,
    ).to(llm.device)

    # 2. GENERATE
    torch.cuda.empty_cache()
    with torch.no_grad():
        output_ids = llm.generate(
            **inputs,
            max_new_tokens=MAX_TOKENS,
            temperature=0.6,
            top_p=0.95,
            top_k=20,
            repetition_penalty=1.0,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    # 3. DECODE
    batch_items = items_to_process[i:i+batch_size]
    batch_responses = []
    for j, out in enumerate(output_ids):
        new_tokens = out[inputs["input_ids"].shape[1]:]
        batch_responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

    # 4. SCORE + SAVE
    for item, response in zip(batch_items, batch_responses):
        is_mcq = bool(item.get("options"))
        gold = item["answer"]
        if is_mcq:
            correct = score_mcq(response, str(gold))
        else:
            gold_list = gold if isinstance(gold, list) else [gold]
            try:
                correct = judger.auto_judge(pred=response, gold=gold_list, options=[[]] * len(gold_list))
            except Exception:
                correct = False

        # 5. SAVE TO FILE
        with open(OUTPUT_PATH, "a") as f:
            f.write(json.dumps({"id": item.get("id"), "is_mcq": is_mcq, "gold": gold, "response": response, "correct": correct}) + "\n")

    # 6. BACKUP TO DRIVE EVERY BATCH
    shutil.copy(OUTPUT_PATH, LOCAL_BACKUP)

print(f"\nGeneration complete! {len(results)} responses generated.")

In [ ]:
# CELL 9: Load results from file and rescore
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
with open(OUTPUT_PATH) as f:
    for line in f:
        results.append(json.loads(line))
print(f"Loaded {len(results)} results")

# Rescore everything fresh
results_rescored = []
for r in results:
    is_mcq = r["is_mcq"]
    gold = r["gold"]
    response = r["response"]
    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(pred=response, gold=gold_list, options=[[]] * len(gold_list))
        except Exception:
            correct = False
    results_rescored.append({**r, "correct": correct})

mcq_res  = [r for r in results_rescored if r["is_mcq"]]
free_res = [r for r in results_rescored if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results_rescored):4d} / {len(results_rescored):4d}  ({acc(results_rescored):.2f}%)")
print("=" * 50)
print("\nBaseline was: 44.76% — compare above!")